# LAB 2 - Azure Services & Shared Lakehouse Setup

This notebook demonstrates the implementation of a governed lakehouse using Azure Data Lake Storage Gen2, Unity Catalog, and Databricks.

The main objectives:
- Configure access to ADLS Gen2 through Unity Catalog
- Ingest raw files into the Bronze layer
- Store data as Delta tables
- Implement incremental ingestion using Auto Loader
- Validate legacy Service Principal access

## 1. Bronze Layer Ingestion

The Bronze layer stores raw source data in Delta format.

Auto Loader is used to ingest CSV files incrementally from ADLS Gen2.

The ingestion process includes:
- Schema tracking
- Checkpointing for idempotency
- Metadata columns for data lineage:
  - source filename
  - ingestion timestamp
  - load date

In [0]:
from pyspark.sql.functions import *

source_path = "abfss://ayyuborujzade@dlspl21databricks.dfs.core.windows.net/raw/gaming/"

schema_path = "abfss://ayyuborujzade@dlspl21databricks.dfs.core.windows.net/schemas/gaming_schema/"

checkpoint_path = "abfss://ayyuborujzade@dlspl21databricks.dfs.core.windows.net/checkpoints/bronze_df/"

In [0]:

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("header", "true")
    .load(source_path)
)


## 2. Data Lineage Columns

Additional metadata columns were added to track ingestion information.

These columns help identify:
- where the file came from
- when it was loaded
- the processing date

In [0]:
bronze_df = (
    bronze_df
    .withColumn(
        "source_filename",
        col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "load_date",
        current_date()
    )
)

## 3. Bronze Delta Table

The streaming DataFrame is written into the Bronze schema as a Delta table.

The checkpoint location allows Auto Loader to remember processed files and prevents duplicate ingestion.

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("dbr_dev.ayyuborujzade_bronze.games")
)

query.awaitTermination()

## 4. Bronze Table Validation

The created Delta table was queried to confirm successful ingestion.

In [0]:
%sql
SELECT *
FROM dbr_dev.ayyuborujzade_bronze.games
LIMIT 10;

## 5. Legacy Service Principal Access

The legacy authentication approach was tested using an Azure Service Principal.

Credentials were retrieved from a Databricks secret scope.

The traditional DBFS mount approach was attempted but blocked by the academy shared cluster because mount operations are not whitelisted.

As an alternative, the same OAuth configuration was applied directly to Spark and ADLS access was validated using an ABFS path.

In [0]:
SECRET_SCOPE = "default2"

tenant_id = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="tenant-id"
)

client_id = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="sp-databricks-adls-appid"
)

client_secret = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key="sp-databricks-adls-appkey"
)

print("Client ID loaded:", bool(client_id))
print("Client secret loaded:", bool(client_secret))
print("Tenant ID loaded:", bool(tenant_id))

In [0]:
storage_account = "dlspl21databricks"
container = "ayyuborujzade"

In [0]:
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

In [0]:
base_path = (
    f"abfss://{container}@"
    f"{storage_account}.dfs.core.windows.net/"
)

display(dbutils.fs.ls(base_path))

## Conclusion

The Bronze layer was successfully implemented using Azure Databricks, Unity Catalog, and Delta Lake.

The final solution includes:
- ADLS Gen2 storage
- Unity Catalog external location
- Bronze schema
- Auto Loader ingestion
- Delta table storage
- Idempotent processing using checkpoints
- Legacy Service Principal authentication demonstration